# Data analysis with pandas (2): combining, I/O and pipelines

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

Real analyses use several tables, store them on disk and clean them before any physics.
Part 2: how to do it with pandas, in a way that can be tested.

**Objectives**

* **combine** tables (`concat`, `merge`, `join`) and **reshape** them (`pivot_table`,
  `melt`, `stack`);
* **plot** tables with matplotlib and `DataFrame.plot`;
* read and write CSV and **HDF5**, and know the other formats (JSON, pickle, Parquet);
* build a cleaning **pipeline** of pure functions with `pipe`, and **test** it;
* keep large tables fast: vectorize, `category`, read only what you need.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pandas.testing as pdt
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 10)

**Recap of [part 1](pandas1.ipynb)**: a DataFrame is a table of columns sharing an index;
select with `.loc` / `.iloc` / masks / `query`, assign in one step with `.loc`;
`groupby` + `agg` or `transform`. We rebuild the same dataset of particles with
`make_events` from the module `fca_events.py`:

In [ ]:
from fca_events import make_events

rng = np.random.default_rng(2025)
events = make_events(2000, rng, missing=0.02)
per_event = events.groupby('event').agg(n=('pt', 'size'), ht=('pt', 'sum'),
                                        charge=('charge', 'sum'))
mass = {'pion': 0.1396, 'kaon': 0.4937, 'muon': 0.1057, 'electron': 0.000511}  # GeV
events.head(3)

## 1. Combining and reshaping

| Function | Use | Key |
|:--|:--|:--|
| `pd.concat` | stack tables, rows or columns | none: aligns on the axis |
| `pd.merge` | SQL-style join | columns (`on`) or index |
| `df.join` | merge on the index | index |

### 1.1 concat

In [ ]:
run1, run2 = make_events(3, rng), make_events(2, rng)
both = pd.concat([run1, run2], keys=[1, 2], names=['run', 'row'])
both.loc[2]                                  # MultiIndex: the rows of run 2

`ignore_index=True` builds a new `RangeIndex` instead. With `axis=1` the tables go side by
side, aligned on the index.

### 1.2 merge

`how` decides which keys survive: `'inner'` (both), `'left'`, `'right'`, `'outer'` (any).
`indicator=True` says where each row comes from.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How many rows?</b></p>

**Decide** how many rows `pd.merge(reco, trigger, on='event', how=...)` has for
`how='inner'`, `'left'` and `'outer'`, before running it.

```python
reco = pd.DataFrame({'event': [1, 2, 3], 'n_tracks': [4, 2, 7]})
trigger = pd.DataFrame({'event': [2, 3, 4], 'trigger': ['mu', 'e', 'mu']})
```

</div>

In [ ]:
reco = pd.DataFrame({'event': [1, 2, 3], 'n_tracks': [4, 2, 7]})
trigger = pd.DataFrame({'event': [2, 3, 4], 'trigger': ['mu', 'e', 'mu']})
print({how: len(pd.merge(reco, trigger, on='event', how=how))
       for how in ['inner', 'left', 'outer']})
pd.merge(reco, trigger, on='event', how='outer', indicator=True)

<details>
<summary><b>Solution</b></summary>

`inner`: 2 (events 2 and 3, in both tables); `left`: 3 (all of `reco`, `trigger` is `NaN`
for event 1); `outer`: 4 (events 1 to 4). With repeated keys the counts grow: each pair
of matching rows gives one row.

</details>

In [ ]:
reco.set_index('event').join(trigger.set_index('event'), how='inner')

### 1.3 Reshaping: pivot_table and melt

`pivot_table` turns a **long** table into a **wide** one, aggregating repeated entries;
`melt` is the inverse.

In [ ]:
counts = pd.pivot_table(events, index='ptype', columns='charge', values='pt',
                        aggfunc='size')
counts

In [ ]:
counts.reset_index().melt(id_vars='ptype', var_name='charge', value_name='n')

* With a MultiIndex: `events.groupby(['ptype', 'charge']).size()` is long; `.unstack()`
  moves the inner level to the columns, `.stack()` back.
* `pivot` is like `pivot_table` but fails if an entry is repeated.

## 2. Visualization

Columns go straight into matplotlib: for publication plots, build the figure yourself.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, group in events.groupby('ptype'):
    axes[0].hist(group['pt'], bins=40, range=(0, 80), histtype='step', label=name)
axes[0].set(xlabel='$p_T$ [GeV]', ylabel='particles', yscale='log')
axes[0].legend()
ok = events.dropna(subset=['eta'])
h = axes[1].hist2d(ok['eta'], ok['phi'], bins=40, cmap='viridis')
fig.colorbar(h[3], ax=axes[1], label='particles')
axes[1].set(xlabel=r'$\eta$', ylabel=r'$\phi$')
plt.tight_layout()
plt.show()

For a quick look, `DataFrame.plot` is a front-end to matplotlib: one line per table.

In [ ]:
counts.plot.bar(ylabel='particles', title='particles by type and charge')
plt.show()

## 3. Reading and writing

| Format | + | − |
|:--|:--|:--|
| CSV | universal, readable | big, slow, loses the dtypes |
| JSON | readable, nested data | bigger; rounds floats (10 digits) |
| pickle | any Python object, exact | Python only; **unsafe**: runs code on load |
| HDF5 | many tables, compression, partial reads | needs PyTables |
| Parquet | columnar, compressed, standard | needs `pyarrow` |

Each has a pair `df.to_<format>(path)` / `pd.read_<format>(path)`.

We write into a temporary directory, removed at the end of the section:

In [ ]:
tmpdir = tempfile.TemporaryDirectory()
path = Path(tmpdir.name)
events.to_csv(path / 'events.csv', index=False)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A CSV round trip</b></p>

**Decide**: after `back = pd.read_csv(path / 'events.csv')`, what is the dtype of
`back['ptype']`? Is `back` equal to `events`?

</div>

In [ ]:
back = pd.read_csv(path / 'events.csv')
print(back['ptype'].dtype, events['ptype'].dtype)
print(back.equals(events), np.allclose(back['pt'], events['pt']))

<details>
<summary><b>Solution</b></summary>

CSV stores text: `ptype` comes back as `str`, not `category`, so the tables are not
equal, even if the numbers are. The dtypes (and an index other than the default) must be
restored by hand, e.g. `pd.read_csv(..., dtype={'ptype': 'category'})`.

</details>

### 3.1 HDF5

A binary format common in physics (also read by `h5py`, C++, Julia…). pandas uses the
**PyTables** package (`tables`; `pytables` in conda):

* **several tables in one file**, each under a key, like a small file system;
* **compression** (`complevel`, `complib`);
* **partial reads**: with `format='table'`, the `data_columns` can be queried with
  `where=`, without loading the whole table.

In [ ]:
h5 = path / 'run.h5'
events.to_hdf(h5, key='events', format='table', data_columns=['pt', 'ptype'],
              complevel=5, complib='zlib')
per_event.to_hdf(h5, key='per_event', format='table', complevel=5, complib='zlib')
with pd.HDFStore(h5, mode='r') as store:
    print(store.keys())          # /events/meta/...: the categories of ptype

In [ ]:
hard_muons = pd.read_hdf(h5, 'events', where='pt > 50 & ptype == "muon"',
                         columns=['event', 'pt', 'eta'])
print(hard_muons.shape)
print(pd.read_hdf(h5, 'per_event').shape)

In [ ]:
for name in ['events.csv', 'run.h5']:
    print(f'{name:12s} {(path / name).stat().st_size / 1e3:6.0f} kB')

`run.h5` holds both tables, the dtypes and an index for `where` queries, and is still
smaller than the CSV of a single table (here; the gain depends on the data).

In [ ]:
tmpdir.cleanup()                 # end of section 3: remove the temporary files

***[+] Lookout.*** In data science the standard columnar format is **Parquet** (`to_parquet`,
`read_parquet`), which needs `pyarrow`.

## 4. A cleaning pipeline

Real tables are messy. A run logbook typed by hand:

In [ ]:
n = 10
log = pd.DataFrame({
    'run': np.arange(100, 100 + n),
    'detector': rng.choice(['tracker', 'calo'], n),
    'temperature': rng.normal(20., 1.5, n).round(1).astype(str),
    'hv': rng.normal(1500., 20., n).round(0).astype(str)})
log.loc[[2, 7], 'temperature'] = 'unknown'
log.loc[5, 'hv'] = '-'
log.loc[8, 'hv'] = '-1497.0'                   # a sign error
log

### 4.1 One step, one function

Each step is a **pure** function: DataFrame in, new DataFrame out; the input is never
modified.

In [ ]:
def to_numbers(df):
    """Convert the measurements to float; invalid entries become NaN."""
    out = df.copy()
    for col in ['temperature', 'hv']:
        out[col] = pd.to_numeric(out[col], errors='coerce').astype(float)
    out.loc[out['hv'] < 0, 'hv'] = np.nan          # unphysical
    return out

def impute(df):
    """Fill the missing measurements with the median of the column."""
    return df.fillna({c: df[c].median() for c in ['temperature', 'hv']})

`df.pipe(f, *args)` is `f(df, *args)`: a chain `log.pipe(a).pipe(b)` reads left to
right, in the order of the steps, instead of `b(a(log))`.

In [ ]:
def standardize(df, col):
    """Add the column col_std = (x - mean) / std."""
    x = df[col]
    return df.assign(**{f'{col}_std': (x - x.mean()) / x.std(ddof=0)})

clean = log.pipe(to_numbers).pipe(impute).pipe(standardize, 'hv')
clean.head()

`ddof=0` divides by $N$ (the numpy default); pandas' `std` divides by $N-1$ (`ddof=1`)
unless told otherwise. No need for scikit-learn to standardize.

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Filling the gaps</b></p>

`impute` replaces a missing temperature by the median. Other options: `0`, the mean,
dropping the row, the previous run (`ffill`), keeping `NaN`.

* What does each one do to the mean and to the spread of the column?
* Should the cleaned table say which values were imputed?

</div>

<details>
<summary><b>Solution</b></summary>

`0` is wrong here: it is not a temperature, and it biases mean and spread. The mean is
pulled by outliers; the median is robust. Any constant shrinks the spread. Dropping rows
is honest but loses the other columns of the row. `ffill` makes sense for a slowly
varying quantity in time order. Imputed values look like measurements: add a flag column,
e.g. `temperature_imputed = df['temperature'].isna()`, *before* filling.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — A broken step</b></p>

This version of `to_numbers` has two bugs. Find them.

```python
def to_numbers(df):
    for col in ['temperature', 'hv']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df[df['hv'] < 0]['hv'] = np.nan
    return df
```

</div>

<details>
<summary><b>Solution</b></summary>

1. It **modifies its input**: `log` itself would be converted, so the step cannot be run
   twice nor compared with the raw data. Start with `out = df.copy()` (or use `assign`).
2. `df[df['hv'] < 0]['hv'] = np.nan` is a **chained assignment**: it changes a temporary
   copy, never `df` (pandas warns with `ChainedAssignmentError`). The negative voltage
   survives. Use `df.loc[df['hv'] < 0, 'hv'] = np.nan`.

</details>

### 4.2 Testing a pipeline

`pandas.testing.assert_frame_equal` compares values, dtypes, index and columns, and says
where two tables differ. Test each step on a tiny table:

In [ ]:
raw = pd.DataFrame({'temperature': ['20.5', 'unknown'], 'hv': ['-1500', '1490']})
expected = pd.DataFrame({'temperature': [20.5, np.nan], 'hv': [np.nan, 1490.]})
pdt.assert_frame_equal(to_numbers(raw), expected)
pdt.assert_frame_equal(raw, pd.DataFrame({'temperature': ['20.5', 'unknown'],
                                          'hv': ['-1500', '1490']}))   # input intact
print('to_numbers: ok')

A failing test says what differs. This one fails on purpose:

In [ ]:
pdt.assert_frame_equal(to_numbers(raw), raw)    # fails on purpose: dtypes differ

## 5. Performance and scaling

* **Vectorize**: column operations, not loops or `apply(axis=1)`.
* Use **`category`** for repeated labels.
* `query` / `eval` for long expressions on large tables.
* Read only the columns you need (`usecols` in `read_csv`).

`memory_usage(deep=True)` gives the bytes of a column, including the text it points to:

In [ ]:
as_str = events['ptype'].astype(str).memory_usage(deep=True)
as_cat = events['ptype'].memory_usage(deep=True)
print(f'str: {as_str / 1e3:.0f} kB, category: {as_cat / 1e3:.0f} kB')

The gap depends on how `str` is stored: Python objects (here) or `pyarrow`, more compact.
`category` stays smaller: one small integer per row.

***[+] Lookout.*** A file too large for memory can be read in **chunks**:
`for chunk in pd.read_csv(path, chunksize=100_000): ...` processes one piece at a time
(e.g. accumulate sums or histograms).

***[+] Lookout.*** For tables larger than memory, or to use all the cores, look at
[**Dask**](https://docs.dask.org) (pandas-like, parallel and out-of-core) and
[**Polars**](https://pola.rs) (a fast DataFrame library with lazy queries). Optional: time
the `groupby('event')` of part 1 with pandas and with Polars on `make_events(10**6, rng)`
(about 5 million particles).

## 6. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Clean a sensor table</b></p>

The cell below builds a messy table of detector sensors.

1. Inspect it: shape, dtypes, `describe`. Which entries are invalid?
2. Replace `'-'`, `'unknown'` and negative currents by `NaN`, convert to float, and decide
   for each column whether to impute or drop.
3. (Optional) Encode `location` with `pd.factorize` and with `pd.get_dummies`.
4. Add `log_current` and `is_hot` (temperature above 25 °C).
5. (Optional) Compute the correlation matrix of the numerical columns and plot it with
   `imshow`; plot current vs temperature.

</div>

In [ ]:
k = 200
sensors = pd.DataFrame({
    'location': rng.choice(['barrel', 'endcap', 'forward'], k),
    'temperature': rng.normal(22., 2.5, k).round(2)})
sensors['current'] = (0.5 * np.exp(0.08 * sensors['temperature'])
                      * rng.lognormal(0., 0.1, k)).round(3)
sensors = sensors.astype({'temperature': str, 'current': str})
sensors.loc[rng.choice(k, 8, replace=False), 'temperature'] = 'unknown'
sensors.loc[rng.choice(k, 6, replace=False), 'current'] = '-'
sensors.loc[rng.choice(k, 4, replace=False), 'current'] = '-0.9'

# your code here

<details>
<summary><b>Solution</b></summary>

```python
s = sensors.copy()
for col in ['temperature', 'current']:
    s[col] = pd.to_numeric(s[col], errors='coerce')   # '-', 'unknown' -> NaN
s.loc[s['current'] < 0, 'current'] = np.nan
print(s.isna().sum())
s = s.dropna()                       # few rows; imputing would hide the correlation
codes, labels = pd.factorize(s['location'])
s = s.assign(location_code=codes, log_current=np.log(s['current']),
             is_hot=s['temperature'] > 25.)
s = pd.concat([s, pd.get_dummies(s['location'], prefix='loc')], axis=1)
corr = s[['temperature', 'current', 'log_current', 'location_code']].corr()
print(corr.round(2))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
axes[0].set_xticks(range(4), corr.columns, rotation=45)
axes[0].set_yticks(range(4), corr.columns)
fig.colorbar(im, ax=axes[0])
axes[1].scatter(s['temperature'], s['current'], c=s['is_hot'], s=10)
axes[1].set(xlabel='temperature [°C]', ylabel='current')
plt.tight_layout()
plt.show()
```

`log_current` is linear in the temperature: their correlation is high (about 0.9, limited
by the noise of the current). The location carries no information (`location_code` is
uncorrelated).

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Concatenate files</b></p>

Inside a `with tempfile.TemporaryDirectory() as tmp:` block:

1. Write three CSV files `part1.csv`, `part2.csv`, `part3.csv` with 5 runs each, with
   **different run numbers** in each file (100–104, 105–109, 110–114) and **overlapping
   but different** columns, e.g. `run, temperature, hv`; `run, hv, detector`;
   `run, Temperature, hv` (a different spelling).
2. Read them back and concatenate them with `pd.concat(..., ignore_index=True)`.
3. Where do the `NaN` come from? Fix the column names and check that each run appears
   once.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
parts = [pd.DataFrame({'run': 100 + np.arange(5) + 5 * i,
                       'hv': rng.normal(1500., 20., 5).round(0)}) for i in range(3)]
parts[0]['temperature'] = rng.normal(20., 1., 5).round(1)
parts[1]['detector'] = 'calo'
parts[2]['Temperature'] = rng.normal(20., 1., 5).round(1)
with tempfile.TemporaryDirectory() as tmp:
    for i, part in enumerate(parts, start=1):
        part.to_csv(Path(tmp) / f'part{i}.csv', index=False)
    read = [pd.read_csv(Path(tmp) / f'part{i}.csv') for i in range(1, 4)]
runs = pd.concat(read, ignore_index=True)
print(runs.isna().sum())          # each column missing in some file gives NaN
runs = pd.concat([r.rename(columns=str.lower) for r in read], ignore_index=True)
assert runs['run'].is_unique
runs
```

`concat` aligns by column name: a column absent from a file is `NaN` in its rows, and
`Temperature` and `temperature` are different columns until the names are normalized.

</details>

## 7. Summary

* `concat` stacks tables, `merge` / `join` match rows by key (`how` decides which keys
  survive); `pivot_table` / `melt` / `stack` / `unstack` reshape long ↔ wide.
* Columns go straight into matplotlib; `DataFrame.plot` for a quick look.
* CSV loses the dtypes; **HDF5** keeps them, stores many compressed tables per file and
  reads only the rows that pass `where`. Never load an untrusted pickle.
* Write cleaning steps as pure functions, chain them with `pipe`, test them with
  `assert_frame_equal`.
* Large tables: vectorize, `category`, only the needed columns; chunks, Dask, Polars.

<details>
<summary><b>[i] Cheat sheet — pandas (2)</b></summary>

| | |
|:--|:--|
| `pd.concat([a, b], keys=, ignore_index=)` | stack |
| `pd.merge(a, b, on=, how=, indicator=)`, `a.join(b)` | match by key |
| `pivot_table`, `melt`, `stack`, `unstack` | reshape |
| `df.plot.bar()`, `plt.hist(df['c'])` | plot |
| `to_csv` / `read_csv(usecols=, dtype=)` | CSV |
| `to_hdf(format='table', data_columns=)` / `read_hdf(where=)` | HDF5 |
| `df.pipe(f, *args)` | chain functions |
| `pdt.assert_frame_equal(a, b)` | testing |
| `memory_usage(deep=True)`, `astype('category')` | memory |

</details>

### Check yourself

1. In `pd.merge(a, b, on='k', how='outer', indicator=True)`, which rows have
   `_merge == 'left_only'`?
2. Why can a CSV round trip change a DataFrame? Name two advantages of HDF5.
3. Why should a pipeline step not modify its input?
4. What does `ddof` change in `std`?

<details>
<summary><b>Solutions</b></summary>

1. Those whose key `k` is only in `a`.
2. CSV stores text: dtypes such as `category` are lost (and JSON rounds floats). HDF5 keeps
   the dtypes, stores many tables in one compressed file, and can read only the rows that
   pass a `where` condition.
3. The raw data must survive: to run the step again, to compare before and after, and to
   test it. `assign` is cheap: under Copy-on-Write the unchanged columns are shared, not
   copied.
4. The divisor: $N - \text{ddof}$. pandas uses `ddof=1` by default, numpy `ddof=0`.

</details>